In [1]:
import numpy as np
import copy
import torch.nn as nn
from tqdm import tqdm
from torch.distributions.multivariate_normal import MultivariateNormal
import torch
from torch import optim
import torch.optim.lr_scheduler as lr_sched
import time
import torch
import os
from dotenv import load_dotenv; load_dotenv()
%cd {os.getenv('PROJECT_PATH')}

from load import load_aggregate_datamed, load_datamed
from input import get_params
from Experiment import Experiment, prepare_experiments
from Utils.MMD_utils import init_params
#from Core.train_node import Trainer
from Core.node import NeuralODE

/home/id94ebid/projects/continuous_GMM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/id94ebid/projects/continuous_GMM


In [48]:
B, T, F, K = 16, 100, 3, 5
dt = 1/(T-1)
ts = torch.linspace(0, 1, T)
X = torch.randn((B, T, F))

In [54]:
def rbf_fkernel(x1, x2, dt, sigma=1.0): # (B, T, F)
    y = torch.square(x1-x2).sum(-1) # (B, T)
    norm = dt * (y[..., 0] / 2 + y[..., 1:-1].sum(dim=-1) + y[..., -1] / 2)
    return torch.exp(-1/(2*sigma**2)*norm)

def rbf_kernel(x1, x2, sigma=1.0): # (B, F) or (B)
    norm = torch.square(x1-x2).sum(-1) # (B, T)
    return torch.exp(-1/(2*sigma**2)*norm)

In [55]:
m = torch.randn((K, T))
pi = torch.full((K,), 1/K)
kernels = [lambda x1, x2: rbf_kernel(x1, x2, sigma) for sigma in np.logspace(-2,1, K)]

TypeError: randn(): argument 'size' (position 1) must be tuple of ints, not tuple

In [ ]:
K_mat = torch.stack([kernel(ts[None, :, None], ts[:, None, None]) for kernel in kernels])

TypeError: rbf_kernel() takes from 2 to 3 positional arguments but 4 were given

In [50]:
K.shape

torch.Size([5])

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
options, params_data, params_optim, params_uq, params_node = get_params()
exps = prepare_experiments(params_data, params_optim, params_uq, params_node, options)

[INFO] Created new run folder → Results_TimeSeries/run1
[INFO] Saved generated data summary to /home/id94ebid/projects/continuous_GMM/Results_TimeSeries/run1/data.csv


In [3]:
combo, data, gamma = exps["exp_1"].values()
data2 = torch.tensor(np.array(data), dtype=torch.float32)
print(data2.shape)

torch.Size([11, 5000, 2])


In [ ]:
# Setup time grids
t_i = torch.linspace(0.0, combo["T"], combo["t_steps"])
t_grid = torch.linspace(0.0, combo["T"], int(combo["T"] / combo["dt"]) + 1)
d = options['fixed_params']['d']
K = 3 #combo["K"]

In [ ]:
local_weights, means, covs = init_params(data2.reshape(-1, d), K=K)

In [ ]:
# observation tensor (T, 1, K)
t_i = torch.linspace(0.0, combo["T"], combo["t_steps"], device=device)
t_grid = torch.linspace(0.0, combo["T"], int(combo["T"] / combo["dt"]) + 1, device=device)

In [ ]:
class LogLikelihoodLoss(nn.Module):
    """
    Log-likelihood loss for mixture of Gaussians.
    
    Implements: -∑_{n=1}^N log ∑_{s=1}^K α_s(t_n;θ) φ(x_n | m_s, Σ_s)
    """
    
    def __init__(self, K, state_dim, device, eps=1e-8):
        """
        Initialize log-likelihood loss.
        
        Args:
            K: Number of mixture components
            state_dim: Dimensionality of state space
            device: Device for computations
            eps: Small constant for numerical stability
        """
        super().__init__()
        self.K = K
        self.state_dim = state_dim
        self.device = device
        self.eps = eps
        
        # Initialize mixture parameters
        # Means: [K, state_dim]
        self.means = nn.Parameter(torch.randn_like(torch.tensor(means, dtype=torch.float32)))

        # Covariances: Use Cholesky decomposition for positive definiteness
        # Shape: [K, state_dim, state_dim]
        self.L_chol = nn.Parameter(torch.tensor(covs, dtype=torch.float32))
    
    def get_covariances(self):
        """Get covariance matrices from Cholesky decomposition."""
        # Ensure lower triangular
        L = torch.tril(self.L_chol)
        # Add small diagonal for numerical stability
        L = L + self.eps * torch.eye(self.state_dim, device=self.device).unsqueeze(0)
        # Σ = L @ L^T
        return torch.bmm(L, L.transpose(-2, -1))
    
    def forward(self, x_pred, x_obs):
        """
        Compute negative log-likelihood loss.
        
        Args:
            x_pred: Predicted trajectories [n_timesteps, batch_size, state_dim]
            x_obs: Observed trajectories [n_timesteps, batch_size, state_dim] 
            t_eval: Evaluation times [n_timesteps]
        
        Returns:
            Negative log-likelihood loss
        """
        n_timesteps, batch_size, state_dim = x_obs.shape

        # Get covariance matrices
        covariances = self.get_covariances()  # [K, state_dim, state_dim]
        
        all_log_probs = []  # shape [N, K]

        for k in range(self.K):
            mean_k = self.means[k]
            cov_k = covariances[k]
            mvn = MultivariateNormal(mean_k, cov_k)
            log_prob_k = mvn.log_prob(x_obs)  # shape [N]
            log_alpha_k = torch.log(x_pred[..., k] + self.eps)  # shape [N]
            all_log_probs.append(log_alpha_k + log_prob_k)

        # Stack into [N, K]
        all_log_probs = torch.stack(all_log_probs, dim=-1)

        # Log-sum-exp across K, then sum across N
        total_log_likelihood = torch.logsumexp(all_log_probs, dim=-1).mean(axis=1).sum()

        # Negative log-likelihood
        return -total_log_likelihood


class Trainer:
    """
    Trainer class for optimizing Neural ODE models with log-likelihood loss.
    
    Minimizes negative log-likelihood between predicted and observed trajectories.
    """
    
    def __init__(self, model, optimizer, scheduler, device, loss_str="LogLikelihood", 
                 reg_lambda=0.0, print_freq=10, max_epochs=1000, 
                 tol=1e-6, tolrel=1e-4, patience=200, K=3):
        """
        Initialize trainer.
        
        Args:
            model: Neural ODE model
            optimizer: PyTorch optimizer
            scheduler: Learning rate scheduler  
            device: Training device
            loss_str: Loss function ("MSE", "L1", "LogLikelihood")
            reg_lambda: L2 regularization coefficient
            print_freq: Print frequency for training progress
            max_epochs: Maximum training epochs
            tol: Absolute tolerance for early stopping
            tolrel: Relative tolerance for early stopping
            patience: Epochs to wait before early stopping
            K: Number of mixture components (for LogLikelihood loss)
        """
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        
        # Configure loss function
        if loss_str is None or loss_str == 'MSE':
            self.loss_func = nn.MSELoss()
        elif loss_str == 'L1':
            self.loss_func = nn.L1Loss()
        elif isinstance(loss_str, LogLikelihoodLoss):
            # Determine state dimension from model
            self.loss_func = loss_str
        else:
            raise ValueError(f"Unknown loss function: {loss_str}")
        
        self.reg_lambda = reg_lambda
        self.print_freq = print_freq
        self.max_epochs = max_epochs
        self.tol = tol
        self.tolrel = tolrel
        self.patience = patience
        
        self.best_loss = float('inf')
        self.best_state = None
        self.loss_history = []

    
    def _compute_regularization(self):
        """Compute L2 regularization terms."""
        reg_loss = 0.0
        # Regularize model parameters
        #reg_loss += self.reg_lambda * sum(torch.sum(param ** 2) for param in self.model.parameters())
        # Regularize mixture parameters if using log-likelihood
        return reg_loss
    
    def _compute_total_loss(self, x_pred, x_obs, t_eval=None):
        """Compute total loss including regularization."""
        loss = self.loss_func(x_pred, x_obs)
        
        reg_loss = self._compute_regularization()
        return loss + reg_loss
    
    def _should_stop_early(self, current_loss, prev_loss, patience_counter):
        """Check early stopping conditions."""
        # Absolute tolerance
        if current_loss < self.tol:
            return True, f"absolute tolerance ({self.tol})"
        
        # Patience exceeded
        if patience_counter >= self.patience:
            return True, f"no improvement in {self.patience} epochs"
        
        # Relative tolerance
        if prev_loss is not None:
            abs_diff = abs(current_loss - prev_loss)
            rel_diff = abs_diff / (abs(prev_loss) + 1e-12)
            if rel_diff < self.tolrel:
                return True, f"relative tolerance ({self.tolrel})"
        
        return False, None
    
    def train(self, t_train, x_obs, initial_state, batch_size=32, shuffle=True):
        """
        Train the Neural ODE model using mini-batch learning.
        
        Args:
            t_train: Time points tensor [n_timesteps]
            x_obs: Observed trajectory [n_timesteps, n_samples, state_dim]
            initial_state: Initial state [n_samples, state_dim]
            batch_size: Size of mini-batches (default: 32)
            shuffle: Whether to shuffle data between epochs (default: True)
        
        Returns:
            Best training loss
        """
        self.model.train()
        
        # Move data to device
        t_train = t_train.to(self.device)
        x_obs = x_obs.to(self.device)
        initial_state = initial_state.to(self.device)
        
        n_samples = x_obs.shape[1]  # Number of samples/trajectories
        n_batches = (n_samples + batch_size - 1) // batch_size  # Ceiling division
        
        prev_loss = None
        patience_counter = 0
        start_time = time.time()
        
        for epoch in range(self.max_epochs):
            epoch_losses = []
            
            # Create indices for this epoch
            if shuffle:
                indices = torch.randperm(n_samples, device=self.device)
            else:
                indices = torch.arange(n_samples, device=self.device)
            
            # Mini-batch training loop
            with tqdm(total=n_batches, desc=f"Epoch {epoch+1}") as pbar: 
                for batch_idx in range(n_batches):
                    # Get batch indices
                    start_idx = batch_idx * batch_size
                    end_idx = min(start_idx + batch_size, n_samples)
                    batch_indices = indices[start_idx:end_idx]
                    
                    # Extract mini-batch data
                    x_obs_batch = x_obs[:, batch_indices, :]  # [n_timesteps, batch_size, state_dim]
                    #initial_state_batch = initial_state[batch_indices, :]  # [batch_size, state_dim]
                    
                    # Zero gradients
                    self.optimizer.zero_grad()
                    
                    # Forward pass
                    initial_state_batch = torch.zeros(len(batch_indices), K)
                    x_pred_batch = self.model(initial_state_batch, eval_times=t_train)
                    batch_loss = self._compute_total_loss(x_pred_batch, x_obs_batch)
                    
                    # Backward pass
                    batch_loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)
                    self.optimizer.step()
                    
                    epoch_losses.append(batch_loss.item())
                    pbar.set_postfix({'Batch': f'{batch_idx+1}/{n_batches}', 'Loss': f'{batch_loss.item():.6f}'})
                    pbar.update(1)
            
            # Calculate average epoch loss
            current_loss = sum(epoch_losses) / len(epoch_losses)
            self.loss_history.append(current_loss)
            
            # Update best model
            if current_loss < self.best_loss:
                self.best_loss = current_loss
                self.best_state = copy.deepcopy(self.model.state_dict())
                # Also save mixture parameters if using log-likelihood
                patience_counter = 0
            else:
                patience_counter += 1
            
            # Update learning rate
            if self.scheduler is not None:
                self.scheduler.step(current_loss)
            
            # Print progress
            if epoch % self.print_freq == 0:
                lr = self.optimizer.param_groups[0]['lr']
                print(f"Epoch {epoch:4d} - {current_loss:.10f} - LR: {lr:.6f} - Batches: {n_batches}")
            
            # Check early stopping
            should_stop, reason = self._should_stop_early(current_loss, prev_loss, patience_counter)
            if should_stop:
                print(f"Early stopping at epoch {epoch} due to {reason}.")
                break
            
            prev_loss = current_loss
        
        # Load best model state
        if self.best_state is not None:
            self.model.load_state_dict(self.best_state)
        
        fit_time = time.time() - start_time
        print(f"Training completed. Best Loss: {self.best_loss:.10f}")
        print(f"Fitting time: {fit_time:.4f} sec")
        print(f"Total samples: {n_samples}, Batch size: {batch_size}, Batches per epoch: {n_batches}")
        
        return self.best_loss

In [ ]:
loss = LogLikelihoodLoss(K, d, device)
model = NeuralODE(
        K, combo["hidden_dim"],
        non_linearity = 'tanh', #combo['non_linearity'],
        N_layers      = 1, #combo['N_layers'],
        T             = combo["T"],
        step_size     = combo["dt"],
        method        = 'euler' #combo["method"]
    ).to(device)
print(">> Using standard Neural ODE with post-hoc normalization")

>> Using standard Neural ODE with post-hoc normalization


In [ ]:
opt  = optim.Adam(list(model.parameters()) + list(loss.parameters()), lr=1e-4)
sch  = lr_sched.ReduceLROnPlateau(opt, mode="min",
                                factor=0.5, patience=50)
sch = None
trainer = Trainer(model, opt, sch, device,
                loss_str     = loss, #combo["loss_func"],
                reg_lambda   = combo["lambda_node"],
                print_freq   = 1, #combo["print_freq"],
                max_epochs   = 20, #combo["max_epochs"],
                tol          = combo["tol_abs"],
                tolrel       = combo["tol_rel"])

trainer.train(t_i, data2, data2[0], batch_size=64)
loss_history = trainer.loss_history

Epoch 1: 100%|██████████| 79/79 [00:03<00:00, 24.93it/s, Batch=79/79, Loss=64.677933]       


Epoch    0 - 63.7635569995 - LR: 0.001000 - Batches: 79


Epoch 2: 100%|██████████| 79/79 [00:05<00:00, 14.99it/s, Batch=79/79, Loss=64.103935]


Epoch    1 - 63.2342998650 - LR: 0.001000 - Batches: 79


Epoch 3: 100%|██████████| 79/79 [00:05<00:00, 15.57it/s, Batch=79/79, Loss=64.720795]


...

Epoch   17 - 58.3880494999 - LR: 0.001000 - Batches: 79


Epoch 19: 100%|██████████| 79/79 [00:05<00:00, 14.39it/s, Batch=79/79, Loss=59.549042]


Epoch   18 - 58.1495808469 - LR: 0.001000 - Batches: 79


Epoch 20: 100%|██████████| 79/79 [00:05<00:00, 13.93it/s, Batch=79/79, Loss=58.503365]

Epoch   19 - 57.9267548670 - LR: 0.001000 - Batches: 79
Training completed. Best Loss: 57.9267548670
Fitting time: 107.9599 sec
Total samples: 5000, Batch size: 64, Batches per epoch: 79
